<h1>LLaVA Inference Testing</h1>

In [7]:
import lmstudio as lms

with lms.Client() as client:
    model = client.llm.model("llava-llama-3-8b-v1_1")
    result = model.respond("Say 'Hello World'")
    print(result)

Hello, world!


In [8]:
from pycocotools.coco import COCO
import random
import os

image_dir = "data/val2017"
caption_dir = "data/captions_val2017.json"
instance_dir = "data/instances_val2017.json"

coco_caps = COCO(caption_dir)
coco_inst = COCO(instance_dir)

img_ids = list(coco_inst.imgs.keys())

random.seed(42)
img_ids = random.sample(img_ids, 200)

dataset = []

for img_id in img_ids:

    img_info = coco_inst.loadImgs(img_id)[0]
    file_name = img_info["file_name"]
    image_path = os.path.join(image_dir, file_name)

    # captions
    cap_ids = coco_caps.getAnnIds(imgIds=img_id)
    caps = coco_caps.loadAnns(cap_ids)
    captions = [c["caption"] for c in caps]

    # boxes
    ann_ids = coco_inst.getAnnIds(imgIds=img_id)
    anns = coco_inst.loadAnns(ann_ids)

    objects = []
    for a in anns:
        cat_name = coco_inst.loadCats(a["category_id"])[0]["name"]
        objects.append({
            "category": cat_name,
            "bbox": a["bbox"]
        })

    dataset.append({
        "image_id": img_id,
        "image_path": image_path,
        "captions": captions,
        "objects": objects
    })

print(dataset[0]["image_path"])
print(dataset[0]["captions"][:2])
print(dataset[0]["objects"][:2])

loading annotations into memory...
Done (t=0.04s)
creating index...
index created!
loading annotations into memory...
Done (t=0.25s)
creating index...
index created!
data/val2017\000000301061.jpg
['A person is moving green hay towards an elephant that is inside the back of a white truck.', 'A man pulls an elephant out of a truck.']
[{'category': 'person', 'bbox': [5.75, 284.32, 66.16, 316.41]}, {'category': 'elephant', 'bbox': [128.29, 155.95, 196.63, 252.64]}]


In [9]:
llava_responses = []

with lms.Client() as client:
    model = client.llm.model("llava-llama-3-8b-v1_1")

    for sample in dataset:
        image_path = sample["image_path"]
        image_handle = lms.prepare_image(image_path)
        chat = lms.Chat()
        chat.add_user_message("Give a detailed description of this image please.", images=[image_handle])
        response = model.respond(chat)

        print(response)
        llava_responses.append(response)

The image captures a moment in the life of an elephant, standing tall and majestic within the confines of a truck bed. The elephant's trunk is extended towards the ground, perhaps exploring its surroundings or reaching for something out of frame. Its body is partially obscured by the wooden slats that make up the sides of the truck bed.

The truck itself is white, providing a stark contrast to the elephant's dark skin. It's parked on a dirt road, suggesting a rural setting. In the background, there's a person standing next to the truck, their presence adding a human element to this unusual scene.

The image doesn't contain any discernible text. The relative positions of the objects suggest that the elephant is the main subject of this photo, with the person and the truck serving as secondary elements in the composition. The overall scene is both intriguing and somewhat surreal, as it's not every day one sees an elephant standing inside a truck bed.
In the heart of a bustling city, a da

In [10]:
import torch
from transformers import Blip2Processor, Blip2ForConditionalGeneration
from PIL import Image

blip_responses = []

device = "cuda" if torch.cuda.is_available() else "cpu"

processor = Blip2Processor.from_pretrained("Salesforce/blip2-opt-2.7b")
model = Blip2ForConditionalGeneration.from_pretrained(
    "Salesforce/blip2-opt-2.7b", device_map={"": 0}, dtype=torch.float16)

for sample in dataset:
    prompt = "Question: Give a detailed description of this image please. Answer:"
    inputs = processor(images=[Image.open(sample["image_path"]).convert("RGB")], text=prompt, return_tensors="pt").to(device, torch.float16)
    response_ids = model.generate(**inputs)
    response = processor.batch_decode(response_ids, skip_special_tokens=True)[0].strip()

    print(response)
    blip_responses.append(response)

Loading weights: 100%|██████████| 1247/1247 [00:10<00:00, 116.01it/s, Materializing param=vision_model.post_layernorm.weight]                               


Question: Give a detailed description of this image please. Answer: An elephant is being loaded into a truck
Question: Give a detailed description of this image please. Answer: A man is doing a trick on a skateboard
Question: Give a detailed description of this image please. Answer: A tennis player is walking on the tennis court
Question: Give a detailed description of this image please. Answer: I am a man
Question: Give a detailed description of this image please. Answer: A woman is sitting on a table with a black umbrella, a pair of black shoes, and a
Question: Give a detailed description of this image please. Answer: A table with wine glasses, a bottle of wine, and a plate of bread
Question: Give a detailed description of this image please. Answer: A living room with a sofa, table, and a television
Question: Give a detailed description of this image please. Answer: A man sitting at a table in a train station
Question: Give a detailed description of this image please. Answer: A compu

In [13]:
llava_scores = []
blip_scores = []

with lms.Client() as client:
    model = client.llm.model("openai/gpt-oss-20b")

    for sample, llava_response in zip(dataset, llava_responses):
        chat = lms.Chat()
        chat.add_user_message("Given the ground-truth qualities of an image including basic captions: " + str(sample["captions"]) + " along with bounding boxes: " + str(sample["objects"]) + ", judge this response from a vision model: " + str(llava_response) + " in terms of helpfulness, relevance, accuracy, and level of detail of the response, and assign a score from 1-10 with 10 being the highest. REMEMBER CAPTIONS AND BOUNDING BOXES DONT CAPTURE EVERYTHING. PLEASE OUTPUT THE SCORE ONLY!")
        response = model.respond(chat)

        print(response)
        llava_scores.append(response)

    for sample, blip_response in zip(dataset, blip_responses):
        chat = lms.Chat()
        chat.add_user_message("Given the ground-truth qualities of an image including captions: " + str(sample["captions"]) + " along with bounding boxes: " + str(sample["objects"]) + ", judge this response from a vision model: " + str(blip_response) + " in terms of helpfulness, relevance, accuracy, and level of detail of the response, and assign a score from 1-10 with 10 being the highest. Remember that the captions don't capture everything in the image. REMEMBER CAPTIONS AND BOUNDING BOXES DONT CAPTURE EVERYTHING. PLEASE OUTPUT THE SCORE ONLY!")
        response = model.respond(chat)

        print(response)
        blip_scores.append(response)

<|channel|>analysis<|message|>Need to judge response; likely inaccurate: person moving hay not mentioned, elephant inside truck but no mention of hay or pulling out. Score maybe 4.<|end|><|start|>assistant<|channel|>final<|message|>4
<|channel|>analysis<|message|>We need to judge helpfulness, relevance, accuracy, detail. The captions indicate skateboarder flying, spill, flip, street. Bounding boxes include cars and person and skateboard. Response describes a skateboarder mid-air, sidewalk, blue jeans etc. It mentions cars parked. That matches some bounding boxes (cars). But no mention of spill or parking lot. Also says "skateboard itself black with white wheels". Might be accurate? Hard to know. Probably moderate score 7.<|end|><|start|>assistant<|channel|>final<|message|>7
<|channel|>analysis<|message|>We need to judge model response: It describes tennis player on grass court, blue shirt, yellow racket. Ground truth says man standing on top of a tennis court holding racquet; bounding 

In [14]:
llava_numbered_scores = []
blip_numbered_scores = []

print(len(llava_scores))

for l, b in zip(llava_scores, blip_scores):
    llava_numbered_scores.append(int(str(l).strip().split("<|message|>")[-1]))
    blip_numbered_scores.append(int(str(b).strip().split("<|message|>")[-1]))

print("The average score of LLaVA was: " + str(sum(llava_numbered_scores) / len(llava_numbered_scores)))

print("The average score of BLIP-2 was: " + str(sum(blip_numbered_scores) / len(blip_numbered_scores)))

200
The average score of LLaVA was: 4.485
The average score of BLIP-2 was: 3.645
